In [ ]:
import datetime
import dijible
from plotly import express as plotly_express
import dotenv
dotenv.load_dotenv()


In [ ]:

from datetime import timedelta

import pandas
pandas.options.display.max_rows = 999
pandas.options.display.min_rows = 999
from datetime import tzinfo

In [ ]:
WINDOW_SIZE_DAYS = 7

In [ ]:
nutrient_name = "calcium"

In [ ]:
intake_df = dijible.get_nutrient_log(nutrient_name)

In [ ]:
intake_df["date"] = (intake_df["consumed_at"].dt.tz_convert("America/Los_Angeles") - datetime.timedelta(hours=5)).dt.date
nutrient_by_day = intake_df.groupby("date").sum()

In [ ]:
nutrient_by_day[f"{nutrient_name}_consumed_moving_average"] = nutrient_by_day[nutrient_name][0:-1].rolling(window=14, center=True).mean()

In [ ]:
nutrient_by_day["date"] = nutrient_by_day.index

In [ ]:
# Make a plot of weight loss and calories consumed per day with different y axes
fig = plotly_express.line(nutrient_by_day, x="date", y=[nutrient_name, f"{nutrient_name}_consumed_moving_average"])
# fig.update_yaxes(title_text="Weight loss per day (pounds)")
fig.update_yaxes(title_text=f"{nutrient_name} consumed per day")

In [ ]:
from scipy.stats import spearmanr
import numpy as np

x = np.arange(len(nutrient_by_day[63:]))
y = nutrient_by_day[nutrient_name][63:]

rho, pval = spearmanr(x, y)
print("Spearman correlation:", rho, "P-value:", pval)

In [ ]:
from scipy.stats import linregress


slope, intercept, r_value, p_value, std_err = linregress(x, y)
print("Slope:", slope, "P-value:", p_value)


In [ ]:
data = nutrient_by_day[nutrient_name]

In [ ]:
import Rbeast as rb
import numpy as np
import matplotlib.pyplot as plt

# Apply BEAST algorithm
result = rb.beast(data, season='none')

# Plot the results
rb.plot(result)


In [ ]:
nutrient_by_day[190:]

In [ ]:
cp = result.trend.cp          # Indices of change points
cp_prob = result.trend.cpPr # Posterior probability for each

In [ ]:
cp

In [ ]:
cp_prob

In [ ]:
significant_cp_indices = [cp[i] for i in range(len(cp_prob)) if cp_prob[i] > 0.95]
significant_cp_indices

In [ ]:
import Rbeast as rb
import numpy as np
import matplotlib.pyplot as plt

# Simulated example
y = data

# Fit Rbeast model with linear trends allowed
result = rb.beast(y, season='none', prior=dict(trendOrder=1))

# result.trend.slope: array of posterior mean slope at each point
# result.trend.slopeSD: std deviation of slope at each point

slope = result.trend.slp
slope_sd = result.trend.slpSD

# Compute z-scores (slope / std dev)
z = slope / slope_sd

# Approximate posterior prob that slope > 0 at each time point
from scipy.stats import norm
p_positive_slope = 1 - norm.cdf(0, loc=slope, scale=slope_sd)

# Plot it
plt.plot(p_positive_slope)
plt.axhline(0.95, color='red', linestyle='--')
plt.title("Probability that slope > 0 over time")
plt.xlabel("Time")
plt.ylabel("P(slope > 0)")
plt.show()
